In [1]:
!pip install -q ultralytics faster-coco-eval
!mkdir -p /kaggle/working/data
!cp -r /kaggle/input/datasets/nttgaming/ripvis-cs431/datasets/train /kaggle/working/data/train

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 42.3 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import json
import yaml
import os
from ultralytics import YOLO
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils
import torch
import random

FOLD_CSV = '/kaggle/input/datasets/nttgaming/ripvis-cs431/fold_management.csv'
DATASET_PATH = '/kaggle/working/data/train'
IOU_THRESHOLDS = np.arange(0.40, 0.96, 0.05)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [4]:
def compute_composite_score(gt_json_path, pred_json_path):
    if not os.path.exists(pred_json_path): return {"Score": 0.0}
    
    gt_coco = COCO(gt_json_path)
    with open(pred_json_path) as f: preds = json.load(f)
    pred_coco = gt_coco.loadRes(preds)
    
    # Evaluate SEG
    counts = [[0, 0, 0] for _ in IOU_THRESHOLDS]
    for img_id in gt_coco.getImgIds():
        gt_anns = [a for a in gt_coco.loadAnns(gt_coco.getAnnIds(imgIds=img_id)) if "segmentation" in a]
        pr_anns = sorted([a for a in pred_coco.loadAnns(pred_coco.getAnnIds(imgIds=img_id)) if "segmentation" in a], 
                         key=lambda x: x.get("score", 0), reverse=True)
        
        if not gt_anns and not pr_anns: continue
        
        # IoU Matching logic (RLE)
        gt_rle = [gt_coco.annToRLE(a) for a in gt_anns]
        pr_rle = [gt_coco.annToRLE(a) for a in pr_anns]
        
        if not pr_anns:
            for idx in range(len(IOU_THRESHOLDS)): counts[idx][2] += len(gt_anns)
            continue
        if not gt_anns:
            for idx in range(len(IOU_THRESHOLDS)): counts[idx][1] += len(pr_anns)
            continue

        ious = maskUtils.iou(pr_rle, gt_rle, np.zeros(len(gt_rle)))
        for idx, thr in enumerate(IOU_THRESHOLDS):
            matched_gt = np.zeros(len(gt_anns), dtype=bool)
            tp = fp = 0
            for i in range(len(pr_anns)):
                eligible = (ious[i] >= thr) & (~matched_gt)
                if eligible.any():
                    tp += 1
                    matched_gt[np.argmax(np.where(eligible, ious[i], -1.0))] = True
                else: fp += 1
            counts[idx][0] += tp; counts[idx][1] += fp; counts[idx][2] += int((~matched_gt).sum())

    # Calculate Metrics
    f1_list, f2_list = [], []
    for tp, fp, fn in counts:
        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1_list.append(2*p*r/(p+r) if (p+r)>0 else 0)
        f2_list.append(5*p*r/(4*p+r) if (4*p+r)>0 else 0)

    # Lấy F1 và F2 tại ngưỡng IoU = 0.50 (index 2 trong mảng 0.40, 0.45, 0.50...)
    idx_50 = 2
    f1_50 = f1_list[idx_50]
    f2_50 = f2_list[idx_50]
    
    # Tính mean cho toàn bộ ngưỡng (0.40 đến 0.95)
    f1_4095 = float(np.mean(f1_list))
    f2_4095 = float(np.mean(f2_list))

    # Composite: 0.25*(F1@50 + F2@50 + mean_F1 + mean_F2)
    score = 0.25 * f1_50 + 0.25 * f2_50 + 0.25 * f1_4095 + 0.25 * f2_4095

    return {
        "F1[50]": float(f1_50),
        "F2[50]": float(f2_50),
        "F1[40:95]": float(f1_4095),
        "F2[40:95]": float(f2_4095),
        "Score": float(score)
    }

In [5]:
def run_training_for_fold(fold_num):
    df = pd.read_csv(FOLD_CSV)
    train_df = df[df['fold'] != fold_num]
    val_df = df[df['fold'] == fold_num]

    # Tạo file list cho YOLO
    train_list = f'train_f{fold_num}.txt'
    val_list = f'val_f{fold_num}.txt'
    
    with open(train_list, 'w') as f:
        f.write('\n'.join([os.path.join(DATASET_PATH, 'images', name) for name in train_df['file_name']]))
    with open(val_list, 'w') as f:
        f.write('\n'.join([os.path.join(DATASET_PATH, 'images', name) for name in val_df['file_name']]))

    # Tạo GT JSON cho Fold này để đánh giá
    with open(os.path.join(DATASET_PATH, 'train_with_additional_data.json'), 'r') as f:
        full_coco = json.load(f)
    
    val_fns = set(val_df['file_name'])
    fold_gt = {
        "images": [i for i in full_coco['images'] if i['file_name'] in val_fns],
        "annotations": [],
        "categories": full_coco['categories']
    }
    v_ids = set(i['id'] for i in fold_gt['images'])
    fold_gt['annotations'] = [a for a in full_coco['annotations'] if a['image_id'] in v_ids]
    
    gt_json_fold = f'gt_fold_{fold_num}.json'
    with open(gt_json_fold, 'w') as f:
        json.dump(fold_gt, f)

    # Cấu hình YAML
    yaml_p = f'rip_f{fold_num}.yaml'
    with open(yaml_p, 'w') as f:
        yaml.dump({'path': '/kaggle/working', 'train': train_list, 'val': val_list, 'nc': 1, 'names': ['rip']}, f)

    # Huấn luyện
    model = YOLO('yolov8n-seg.pt')
    !yolo task=segment mode=train \
        model=yolov8n-seg.pt \
        data={yaml_p} \
        epochs=150 imgsz=640 \
        batch=200 \
        workers=2 \
        device=0,1 \
        seed=42 \
        cache=False \
        project=Rip_Project name=fold_{fold_num}
    model = YOLO(f'/kaggle/working/runs/segment/Rip_Project/fold_{fold_num}/weights/best.pt')
    model.val(data=yaml_p, split='val', save_json=True, project='Eval', name=f'fold_{fold_num}', conf=0.25)

    pred_json_path = f'/kaggle/working/runs/segment/Eval/fold_{fold_num}/predictions.json'
    
    # Kiểm tra xem YOLO có sinh ra file không
    if not os.path.exists(pred_json_path):
        print(f"Không tìm thấy {pred_json_path}. Trả về Score 0.0")
        score_results = {"Score": 0.0, "F1[50]": 0.0, "F2[50]": 0.0, "F1[40:95]": 0.0, "F2[40:95]": 0.0}
    else:
        # 1. Tạo từ điển map Tên file -> ID số từ biến fold_gt đã tạo ở trên
        filename_to_id = {os.path.splitext(img['file_name'])[0]: img['id'] for img in fold_gt['images']}
        
        # 2. Mở file JSON dự đoán
        with open(pred_json_path, 'r') as f:
            preds = json.load(f)
            
        # 3. Đổi ID chữ thành ID số
        valid_preds = []
        for p in preds:
            stem = str(p['image_id']) # ví dụ: "RipVIS-141_00060"
            if stem in filename_to_id:
                p['image_id'] = filename_to_id[stem] # Gán lại ID số nguyên
                valid_preds.append(p)
                
        # 4. Ghi đè lại file
        with open(pred_json_path, 'w') as f:
            json.dump(valid_preds, f)
            
        # 5. Đưa file ĐÃ SỬA vào hàm tính điểm
        score_results = compute_composite_score(gt_json_fold, pred_json_path)
        
    score_results['fold'] = fold_num

    json_result_path = f'result_fold_{fold_num}.json'
    with open(json_result_path, 'w') as f:
        json.dump(score_results, f, indent=4)
        
    return score_results

In [6]:
current_fold = 2
result = run_training_for_fold(current_fold)
print(f"Kết quả Fold {current_fold}: {result}")

Ultralytics 8.4.39 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=200, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=rip_f2.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fold_2, nbs=64, nms=False, opset